# winch_launch_simulator v1.2.0

4 canonical cases = 2 plant fidelities × 2 pilot/operation policies.

- RWTH / Prescribed tension + Ideal Secant
- Club / Prescribed tension + Ideal Secant
- RWTH / Dynamic winch + FEM
- Club / Dynamic winch + FEM

旧 Dynamic Secant と Standard SOP は使用しない。Prescribed pair は完全に同じ張力入力を使用する。


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
from IPython.display import Video, display

# Allow this notebook to use the repository source directly.  An editable
# package install (pip install -e .) is not required for notebook use.
ROOT = Path.cwd()
if not (ROOT / "winch_launch_simulator").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "winch_launch_simulator").is_dir():
    raise FileNotFoundError(
        "Open this notebook from the repository root or the notebooks directory."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from winch_launch_simulator.simulation import load_case, run_simulation
from winch_launch_simulator.visualization import plot_summary

ROOT = Path.cwd()
CASE_PATHS = {
    "rwth_prescribed": ROOT / "cases" / "g103a_rwth_reference_prescribed_tension.json",
    "club_prescribed": ROOT / "cases" / "g103a_club_prescribed_tension.json",
    "rwth_fem": ROOT / "cases" / "g103a_rwth_reference_dynamic_winch_fem.json",
    "club_fem": ROOT / "cases" / "g103a_club_dynamic_winch_fem.json",
}


## 1. 実行と評価量


In [ ]:
def event(result, code):
    return next((item for item in result["events"] if item["code"] == code), None)

def metrics(result):
    h = result["history"]
    t = np.asarray(h["time_s"], dtype=float)
    liftoff = event(result, "LIFTOFF")
    release = event(result, "PILOT_COMMAND_RELEASE")
    i0 = 0 if liftoff is None else int(np.argmin(np.abs(t - liftoff["time"])))
    i1 = len(t)-1 if release is None else int(np.argmin(np.abs(t - release["time"])))
    sl = slice(i0, i1+1)
    return {
        "case_id": result["case_id"],
        "release_altitude_m": None if release is None else h["agl_m"][i1],
        "max_eas_m_s": float(np.max(np.asarray(h["eas_m_s"])[sl])),
        "max_tension_n": float(np.max(np.asarray(h["cable_tension_n"])[sl])),
        "min_stall_margin_m_s": float(np.min(np.asarray(h["stall_margin_m_s"])[sl])),
        "max_abs_pitch_rate_deg_s": float(np.max(np.abs(np.asarray(h["pitch_rate_deg_s"])[sl]))),
        "stops": [e["code"] for e in result["events"] if e["severity"] == "stop"],
    }


In [ ]:
results = {}
rows = []
for name, path in CASE_PATHS.items():
    result = run_simulation(load_case(path))
    results[name] = result
    rows.append(metrics(result))
rows


## 2. Prescribed pair

両ケースは同じ Ideal Secant と同じ prescribed tension profile を使用する。差は pilot policy だけである。


In [ ]:
rwth_p = load_case(CASE_PATHS["rwth_prescribed"])
club_p = load_case(CASE_PATHS["club_prescribed"])
assert rwth_p["winch"]["tension_profile"] == club_p["winch"]["tension_profile"]
assert rwth_p["cable"] == club_p["cable"] == {"model": "ideal_secant"}
plot_summary(results["rwth_prescribed"])
plot_summary(results["club_prescribed"])


## 3. Dynamic FEM pair

FEM cable と winch hardware は共通。RWTH は tension control、Club は throttle operation を用いる。


In [ ]:
rwth_f = load_case(CASE_PATHS["rwth_fem"])
club_f = load_case(CASE_PATHS["club_fem"])
assert rwth_f["cable"] == club_f["cable"]
assert rwth_f["winch"]["control"]["mode"] == "tension_control"
assert club_f["winch"]["control"]["mode"] == "club_throttle"
plot_summary(results["rwth_fem"])
plot_summary(results["club_fem"])


## 4. Club pilot commonality


In [ ]:
assert club_p["pilot"] == club_f["pilot"]
club_f["pilot"]["full_climb"]


## 5. Interpretation

Prescribed pair は same forcing 下の pilot-policy comparison。Club Prescribed は実 Club winch operation の再現ではない。実フライト validation の主対象は Club Dynamic winch + FEM とする。
